# Sugestões do Pedro (reunião de 23/10): filtro de 25% e balanceamento

Roda o `analysis/26_filtro_e_balanceamento.py` sobre as tabelas da Paola e do pipeline próprio.

1. **Filtro de colunas**: regra do Pedro (excluir colunas com mais de 25% de nulos) adaptada para microbioma, onde nulo é zero. Compara sem filtro, prevalência ≥ 10% (atual), ≥ 25%, ≥ 50% e ≤ 25% de zeros, sempre removendo colunas constantes e duplicadas. O filtro é calculado só no treino de cada fold.
2. **Balanceamento**: nenhum, class_weight, RandomUnderSampler, SMOTE, BorderlineSMOTE, ADASYN e SMOTETomek, aplicados no treino inteiro (`global`) ou dentro de cada projeto que tem as duas classes (`por_projeto`).

Tudo em câncer x controle, CV aleatória x leave-one-project-out.

**Como usar:** Ambiente de execução → Executar tudo. Na primeira célula o Colab pede para entrar com a conta Google que tem acesso às pastas.

In [ ]:
# 1. Autentica com a sua conta Google (baixa pelo ID, não precisa da pasta estar pública)
from google.colab import auth
auth.authenticate_user()

import io, os
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
drive = build("drive", "v3")

def baixa(file_id, destino):
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    if os.path.exists(destino):
        print("já existe:", destino); return
    req = drive.files().get_media(fileId=file_id, supportsAllDrives=True)
    with io.FileIO(destino, "wb") as f:
        d = MediaIoBaseDownload(f, req, chunksize=50 * 1024 * 1024)
        fim = False
        while not fim:
            _, fim = d.next_chunk()
    print("ok:", destino, round(os.path.getsize(destino) / 1e6, 1), "MB")

In [ ]:
# 2. Código (repositório público) e dependências
!git clone -q https://github.com/tassid/cervical-cancer-microbiome-ml.git /content/repo 2>/dev/null || git -C /content/repo pull -q
!pip install -q imbalanced-learn xgboost h5py

In [ ]:
# 3. Dados processados (IDs dos arquivos nas pastas compartilhadas)
DATA = "/content/data"
baixa("1NOGYus4HM3Q9pSztiqenzT7Hv5T-2auM", f"{DATA}/paola/feature-table.txt")      # Paola: Results/feature-table.txt
baixa("1EFoTgWeLoiOIG78jA2KARLGdoh-L7tIo", f"{DATA}/tassi/metadata_master.tsv")      # Tassi: metadata/metadata_master.tsv
baixa("1ToSEQRXKu0JRNFoHELJ0wQ5AsIbOfzwd", f"{DATA}/tassi/genus_table_final.qza")    # Tassi: qiime/genus_table_final.qza

### P22 reprocessado (opcional, para a tabela `TASSI_V2`)

O arquivo `p22_v3v4_genero.pkl` (saída do `08b_reprocessa_p22.py`) não está no Drive, está na pasta `analysis/results/` do repositório no seu computador. Se quiser rodar também a tabela do seu pipeline com o P22 corrigido, rode a célula abaixo e envie esse arquivo. Se pular, roda só com as tabelas da Paola.

In [ ]:
RES = "/content/results"
os.makedirs(RES, exist_ok=True)
from google.colab import files
enviado = files.upload()   # escolha p22_v3v4_genero.pkl
for nome, conteudo in enviado.items():
    open(f"{RES}/{nome}", "wb").write(conteudo)
print(os.listdir(RES))

In [ ]:
# 4. Quais tabelas rodar
RES = "/content/results"; os.makedirs(RES, exist_ok=True)
tabelas = ["PAOLA", "PAOLA_LIMPA"]
if os.path.exists(f"{RES}/p22_v3v4_genero.pkl"):
    tabelas.append("TASSI_V2")
print("vai rodar:", tabelas)
os.environ.update(DATA_DIR=DATA, OUT_DIR=RES)

In [ ]:
# 5. Roda (alguns minutos por tabela)
import subprocess, sys
for t in tabelas:
    print(f"\n######## {t} ########", flush=True)
    p = subprocess.run([sys.executable, "26_filtro_e_balanceamento.py", t], cwd="/content/repo/analysis",
                       capture_output=True, text=True, env=os.environ)
    print(p.stdout[p.stdout.find("== Parte A"):] if "== Parte A" in p.stdout else p.stdout[-3000:])
    if p.returncode: print(p.stderr[-3000:])

In [ ]:
# 6. Gráficos: AUC na CV aleatória x leave-one-project-out
import pandas as pd, matplotlib.pyplot as plt
for t in tabelas:
    a = pd.read_csv(f"{RES}/26_filtro_{t}.csv")
    b = pd.read_csv(f"{RES}/26_balanceamento_{t}.csv")
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
    ordem = ["sem filtro", "prev>=10%", "prev>=25%", "prev>=50%", "<=25% zeros (Pedro)"]
    pa = a.groupby(["filtro", "esquema"]).AUC.median().unstack().reindex(ordem)
    pa.plot.bar(ax=ax[0], rot=20); ax[0].set_title(f"{t}: filtro de colunas")
    for i, n in enumerate(a.groupby("filtro").n_generos.median().reindex(ordem)):
        ax[0].text(i, 0.02, f"{int(n)} gên.", ha="center", fontsize=8)
    b["metodo"] = b.balanceamento + " / " + b.modo
    b.groupby(["metodo", "esquema"]).AUC.median().unstack().plot.barh(ax=ax[1])
    ax[1].set_title(f"{t}: balanceamento")
    for x in ax: x.axhline(0.5, ls=":", c="gray") if x is ax[0] else x.axvline(0.5, ls=":", c="gray")
    ax[0].set_ylabel("AUC (mediana RF/LogReg)"); plt.tight_layout(); plt.show()

In [ ]:
# 7. Salva os resultados no seu Drive (pasta MyDrive/resultados_26) e baixa um zip
from google.colab import drive as gdrive
gdrive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/resultados_26 && cp {RES}/26_*.csv /content/drive/MyDrive/resultados_26/
!cd {RES} && zip -q -j /content/resultados_26.zip 26_*.csv
files.download("/content/resultados_26.zip")